This pipeline test uses:
- **Count and area:** YOLOv8-seg
- **Stage Classificator:** ResNet-18
- **Action Motor:** Moondream2

In [15]:
from pathlib import Path
import sys

project_root = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents)
     if (parent / "utils" / "utls_install_libs.py").is_file()),
    None,
)

if project_root is None:
    raise FileNotFoundError("Could not find utils/utls_install_libs.py")

sys.path.insert(0, str(project_root))

import os
import sys
import time
import cv2
import pandas as pd
import numpy as np
import torch
from pathlib import Path
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
from torchvision import models, transforms
from transformers import AutoModelForCausalLM, AutoTokenizer
from PIL import Image
from utils.utls_work_images import build_dataset

In [16]:
class Pipeline05:
    def __init__(self, yolo_weights, resnet_weights_path=None):
        # Estágio 1A: YOLOv8
        self.yolo = YOLO(yolo_weights)
        
        # Estágio 2: ResNet-18 (Substituindo a CNN base)
        self.resnet = models.resnet18(weights=None)
        self.resnet.fc = torch.nn.Linear(self.resnet.fc.in_features, 2)
        
        # Carregar os pesos só se tiverem sido fornecidos (após o fine-tuning)
        if resnet_weights_path and os.path.exists(resnet_weights_path):
             self.resnet.load_state_dict(torch.load(resnet_weights_path, map_location=torch.device('cpu')))
             
        self.resnet.eval()
        self.resnet_transforms = transforms.Compose([
            transforms.ToTensor(), # Converte numpy array [0,1] para Tensor
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # Estágio 3: Moondream2 (Motor de Ação)
        model_id = "vikhyatk/moondream2"
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.moondream = AutoModelForCausalLM.from_pretrained(model_id, trust_remote_code=True)
        self.moondream.eval()

In [17]:
def extract_opencv(self, img_array_normalized):
        """Estágio 1B: OpenCV recebendo array normalizado [0, 1]"""
        # Desnormalizar e converter para uint8
        img_uint8 = (img_array_normalized * 255).astype(np.uint8)
        
        # Converter para escala de cinza para processamento clássico
        img_gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
        
        # Suavização e Limiarização
        blurred = cv2.GaussianBlur(img_gray, (5, 5), 0)
        _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        qtd_bolhas = len(contours)
        area_vapor = sum(cv2.contourArea(c) for c in contours)
        area_total = img_gray.shape[0] * img_gray.shape[1]
        vapor_percentual = min((area_vapor / area_total) * 100, 100.0)
        
        return qtd_bolhas, vapor_percentual

In [18]:
def extract_yolo(self, img_array_normalized):
        """Estágio 1A: YOLOv8 recebendo array normalizado [0, 1]"""
        # Desnormalizar e converter para uint8
        img_uint8 = (img_array_normalized * 255).astype(np.uint8)
        
        results = self.yolo.predict(img_uint8, verbose=False)[0]
        
        qtd_bolhas = len(results.boxes) if results.boxes else 0
        vapor_percentual = 0.0
        
        if results.masks is not None:
            # Soma a área de cada máscara identificada
            mask_area = 0
            for mascara in results.masks.data.cpu().numpy():
                 mascara_resized = cv2.resize(mascara, (224, 224))
                 mask_area += np.sum(mascara_resized > 0.5)
            
            vapor_percentual = min((mask_area / (224 * 224)) * 100, 100.0)
            
        return qtd_bolhas, vapor_percentual

In [19]:
def classify_resnet(self, img_array_normalized):
        """Estágio 2: ResNet-18 recebendo array normalizado [0, 1]"""
        # Como o teu array original tem shape (224, 224, 3) e já está / 255.0
        # O ToTensor converte (H, W, C) para (C, H, W)
        input_tensor = self.resnet_transforms(img_array_normalized.astype(np.float32)).unsqueeze(0)
        with torch.no_grad():
            outputs = self.resnet(input_tensor)
            _, predicted = torch.max(outputs, 1)
            
        # O teu y_exp é 0 (Expulsion) e y_inc é 1 (Incubation) de acordo com o utls_work_images.py
        classes = ["Expulsion", "Incubation"]
        return classes[predicted.item()]

In [20]:
def generate_action(self, img_array_normalized, state, count, vapor):
        """Estágio 3: Moondream2 recebendo array normalizado [0, 1]"""
        # Preparar a imagem PIL exigida pelo VLM
        img_uint8 = (img_array_normalized * 255).astype(np.uint8)
        image_pil = Image.fromarray(img_uint8)
        
        prompt = (f"The current thermal state of the electric motor cooling fluid is '{state}'. "
                  f"The vision system detected {count} bubbles, which occupy {vapor:.2f}% of the area. "
                  "Based on this exact data, give a single short sentence stating the immediate action required.")
        
        enc_image = self.moondream.encode_image(image_pil)
        action = self.moondream.answer_question(enc_image, prompt, self.tokenizer)
        return action

In [21]:
def run_comparison_benchmark(self, img_array_normalized):
        """Processa um único array de imagem pelo Pipeline"""
        
        # Extrações Paralelas
        cv_count, cv_vapor = self.extract_opencv(img_array_normalized)
        yolo_count, yolo_vapor = self.extract_yolo(img_array_normalized)
        
        # Classificação ResNet
        state = self.classify_resnet(img_array_normalized)
        
        # Motor Moondream
        action_cv_data = self.generate_action(img_array_normalized, state, cv_count, cv_vapor)
        action_yolo_data = self.generate_action(img_array_normalized, state, yolo_count, yolo_vapor)
        
        return {
            "Estado_ResNet": state,
            "OpenCV_Bolhas": cv_count, 
            "OpenCV_Vapor_%": round(cv_vapor, 2), 
            "OpenCV_Veredito": action_cv_data,
            "YOLOv8_Bolhas": yolo_count, 
            "YOLOv8_Vapor_%": round(yolo_vapor, 2), 
            "YOLOv8_Veredito": action_yolo_data
        }

In [ ]:
if __name__ == "__main__":
    print("📥 [1/4] Baixando matrizes do Google Drive...")
    X_exp, y_exp = build_dataset(limit_per_class_collected=50, folder_name_collected="expulsion")
    X_inc, y_inc = build_dataset(limit_per_class_collected=50, folder_name_collected="incubation")

    X_completo = np.concatenate((X_exp, X_inc), axis=0)
    y_completo = np.concatenate((y_exp, y_inc), axis=0)
    X_train, X_test, y_train, y_test = train_test_split(X_completo, y_completo, test_size=0.2, random_state=42)
    
    print("🧠 [2/4] Preparando o Pipeline 05 e baixando pesos...")
    pipeline = Pipeline05(yolo_weights="yolov8n-seg.pt", resnet_weights_path="resnet_finetuned.pth")

    resultados_p5 = []
    print("⏱️ [3/4] Iniciando cronometragem e processamento do Pipeline 05...")
    
    for i in range(len(X_test)):
        img_array = X_test[i]
        estado_real = "Incubation" if y_test[i] == 1 else "Expulsion"
        
        print(f"  -> Processando imagem {i+1}/{len(X_test)} (Real: {estado_real})")
        
        inicio_timer = time.perf_counter()
        dados = pipeline.run_comparison_benchmark(img_array)
        fim_timer = time.perf_counter()
        
        latencia_ms = round((fim_timer - inicio_timer) * 1000, 2)
        
        linha = {
            "ID_Teste": i,
            "Real": estado_real,
            "Predicao_ResNet18": dados["Estado_ResNet"],
            "OpenCV_Qtd_Bolhas": dados["OpenCV_Bolhas"],
            "OpenCV_Vapor_%": dados["OpenCV_Vapor_%"],
            "OpenCV_Acao_Sugerida": dados["OpenCV_Veredito"],
            "YOLO_Qtd_Bolhas": dados["YOLOv8_Bolhas"],
            "YOLO_Vapor_%": dados["YOLOv8_Vapor_%"],
            "YOLO_Acao_Sugerida": dados["YOLOv8_Veredito"],
            "Latencia_ms": latencia_ms
        }
        
        resultados_p5.append(linha)
        
    df_p5 = pd.DataFrame(resultados_p5)
    df_p5.to_csv('benchmark_pipeline_05.csv', index=False)
    
    print(f"\n✅ [4/4] Benchmark concluído! Salvo como 'benchmark_pipeline_05.csv'.")
    print(f"📊 Latência Média do Pipeline 05: {df_p5['Latencia_ms'].mean():.2f} ms")